# RQ2: native input merge / no-merge pilot

Compare **uncompressed LLaVA576**, **original input ApET288** and **the same native selection+basis without merging** on the fixed **GQA12 + MMStar12 + TextVQA6** RQ1 panel. Decoder compression stays off. This diagnostic cohort does not estimate population accuracy. We keep the existing MMStar quality/grounding flags.

Push the new files to `apet-analysis`, import into an **editable draft**, and enable **Internet + T4 ×2**. Attach the **same private `rq1-ih-inputs-v1` Dataset** used for the input-IH run: the existing **34.1 MiB** `rq1-ih-inputs-v1.zip` or its extracted files. **No new upload is needed if that Dataset already exists.** The older `rq1-inputs-v1` bundle alone is insufficient. A Dataset may remain private.

After a native parity failure, first set **`RUN_PARITY_DIAGNOSTIC = True`** and keep **`RUN_RQ2_PILOT = False`**. This runs only the first question: three raw native encodes, three observed native encodes, two no-merge encodes and one observed native generation, with saved numerical comparisons. It does **not** run or approve the 30-question pilot. Otherwise, set only **`RUN_RQ2_PILOT = True`** for the full pilot after diagnostic review. Leave `QUESTIONS = 30`. Both flags default to False and stop before setup/inference. See `research/RQ2_INPUT_MERGE_PILOT.md` for the exact comparison and reuse instructions.


In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

RUN_RQ2_PILOT = False  # Full 30-question answer pilot, after reviewing parity diagnostics.
RUN_PARITY_DIAGNOSTIC = False  # First-question numerical checks only; keep the pilot flag False.
QUESTIONS = 30         # Fixed full diagnostic panel: GQA12, MMStar12, TextVQA6.
REPO_URL = "https://github.com/doguineyes/CSE590-ApET.git"
REPO_REF = "apet-analysis"
INPUT_ZIP = None       # Optional exact ZIP path OR extracted folder containing references/.
CHECKPOINT_ARGS = []   # Optional ["--model", "/kaggle/input/...", "--vision-model", "/kaggle/input/..."]

if RUN_RQ2_PILOT and RUN_PARITY_DIAGNOSTIC:
    raise ValueError("Enable only one mode: numerical diagnostic OR full 30-question pilot.")
if not (RUN_RQ2_PILOT or RUN_PARITY_DIAGNOSTIC):
    raise RuntimeError("Enable RUN_PARITY_DIAGNOSTIC or RUN_RQ2_PILOT, Internet + T4 x2, then run cells manually.")
if QUESTIONS != 30:
    raise ValueError("Keep QUESTIONS = 30 for the first fixed-panel pilot.")
print("Preparing RQ2 code; model inference has not started.", flush=True)
repo = Path("/kaggle/working/CSE590-ApET")
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True, timeout=120)
subprocess.run(["git", "fetch", "origin"], cwd=repo, check=True, timeout=120)
subprocess.run(["git", "checkout", REPO_REF], cwd=repo, check=True, timeout=120)
if REPO_REF == "apet-analysis":
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True, timeout=120)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo, check=True, timeout=120)
if not (repo / "scripts/run_rq2_input_merge.py").is_file():
    raise RuntimeError("Push the new RQ2 files first, then update this checkout.")
original_python = repo / "environments/original/.venv/bin/python"
rq2_runs = []
print("Repository:", repo, flush=True)


## Prepare existing inputs and locked original runtime

The same original isolated environment and checkpoint cache are reused in a live session. Kaggle's notebook kernel remains unchanged; no restart or default-Transformers downgrade is required. A new session may need to reinstall the environment and download weights again.

The tiny native tests run before pretrained loading. Their complete console output and the runtime report are saved and included as runner evidence. Any skipped native test stops this notebook prerequisite.


In [ ]:
sys.path.insert(0, str(repo / "scripts"))
from rq1_input_ih import locate_inputs
inputs = locate_inputs(
    [Path("/kaggle/input"), Path("/kaggle/working")],
    Path("/kaggle/working/rq1-ih-inputs-v1"), explicit=INPUT_ZIP,
)
print("Accepted input directory:", inputs, flush=True)
setup_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
setup_log = Path(f"/kaggle/working/rq2-setup-{setup_stamp}.log")
runtime_report = Path(f"/kaggle/working/rq2-runtime-{setup_stamp}.json")

def stream_command(command, stream, *, cwd=None):
    print("Running:", " ".join(map(str, command)), flush=True)
    stream.write("COMMAND " + repr(command) + "\n")
    stream.flush()
    process = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, encoding="utf-8", bufsize=1)
    lines = []
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            stream.write(line)
            stream.flush()
            lines.append(line)
        status = process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()
    stream.write(f"EXIT {status}\n")
    stream.flush()
    if status:
        raise RuntimeError(f"Command failed (exit {status}); preserve {stream.name} before changing anything.")
    return "".join(lines)

with setup_log.open("x", encoding="utf-8") as stream:
    stream_command([
        "bash", "bootstrap/setup_original_linux.sh", "--require-cuda", "--expected-gpus", "2",
        "--output", str(runtime_report),
    ], stream, cwd=repo)
    # Explicit import prevents the no-Torch local test-skip path on Kaggle.
    stream_command([str(original_python), "-X", "utf8", "-I", "-c",
                    "import torch; print(torch.__version__)"], stream, cwd=repo)
    native_output = stream_command([
        str(original_python), "-X", "utf8", "-m", "unittest", "research.tests.test_rq2_native",
        "research.tests.test_rq2_parity_diagnostic.DiagnosticPolicyTests",
        "research.tests.test_rq2_parity_diagnostic.DiagnosticTensorTests", "-v",
    ], stream, cwd=repo)
    count = re.search(r"Ran (\d+) tests?", native_output)
    if not count or int(count.group(1)) == 0 or re.search(r"skipped\s*=", native_output):
        raise RuntimeError("Native prerequisites must run at least one test with zero skips; preserve the setup log.")
print("Setup and native prerequisites passed.", flush=True)
print("Setup log:", setup_log)
print("Runtime report:", runtime_report)


## Streaming run helper

In pilot mode one loaded model serves all 30 questions and all three arms. Diagnostic mode loads the same model but checks only the first question with repeated raw/observed/no-merge encodes; it saves feature tensors and exact/numerical comparisons. A completed diagnostic is evidence to review, not a passed pilot. This is ordinary answer generation plus parity/reference checks, not another token-intervention sweep. Native E selection, basis source-to-slot assignment, output order and budget stay matched between the merge/no-merge arms.

The runner archives its evidence on normally handled failures. The wrapper keeps the runner's ZIP untouched, saves a separate console log, and creates a labeled fallback only if no ZIP was produced. Hard session termination may prevent cleanup. Do not pull code or launch another model while this process runs.


In [ ]:
def run_rq2_pilot(*, diagnostic=False):
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    prefix = "rq2-native-parity-diagnostic-v1" if diagnostic else "rq2-input-merge-pilot-v1"
    folder = Path(f"/kaggle/working/{prefix}-{stamp}")
    log = folder.with_suffix(".log")
    archive = folder.with_suffix(".zip")
    rq2_runs.append(folder)  # Keep download paths available even if the run fails.
    command = [str(original_python), "-X", "utf8", "-I", "scripts/run_rq2_input_merge.py",
        "--inputs", str(inputs), "--output", str(folder), "--questions", str(QUESTIONS),
        "--require-cuda", "--expected-gpus", "2", "--download-budget-gib", "20",
        "--setup-log", str(setup_log), "--runtime-report", str(runtime_report), *CHECKPOINT_ARGS]
    if diagnostic:
        command.append("--diagnose-native-parity")
    print("Mode:", "first-question parity diagnostic only" if diagnostic else "full 30-question answer pilot", flush=True)
    print("Run directory:", folder, flush=True)
    fallback_archive = False
    try:
        with log.open("x", encoding="utf-8") as stream:
            stream_command(command, stream, cwd=repo)
    finally:
        # Normally the runner already created this archive, including on handled failure.
        if not archive.exists():
            fallback_archive = True
            folder.mkdir(parents=True, exist_ok=True)
            fallback = folder / "notebook-fallback"
            fallback.mkdir(exist_ok=True)
            (fallback / "README.txt").write_text(
                "Notebook fallback evidence: the process did not produce its normal ZIP.\n"
                "This archive does not establish successful execution or complete results.\n",
                encoding="utf-8")
            for source in [setup_log, runtime_report, log]:
                if source.is_file():
                    shutil.copy2(source, fallback / source.name)
            with zipfile.ZipFile(archive, "x", zipfile.ZIP_DEFLATED) as bundle:
                for source in sorted(folder.rglob("*")):
                    if source.is_file():
                        bundle.write(source, source.relative_to(folder))
            print("Created incomplete fallback evidence ZIP:", archive, flush=True)
        print("Download:", archive, "and", log, flush=True)
        report_path = folder / "report.json"
        if report_path.exists():
            try:
                report = json.loads(report_path.read_text(encoding="utf-8"))
                label = "Diagnostic-only status:" if diagnostic else "Pilot status:"
                print(label, report.get("status"), "wall seconds:", report.get("wall_seconds"))
                names = ["diagnostic_only", "scientific_pilot_passed", "progress", "error"] if diagnostic else ["summaries", "progress", "error"]
                for name in names:
                    if report.get(name) is not None:
                        print(name + ":", report[name])
                if diagnostic and report.get("diagnostic"):
                    detail = report["diagnostic"]
                    print("diagnostic:", {key: detail.get(key) for key in
                        ["case_id", "status", "completion_meaning", "tensor_file"]})
                    print("Comparison groups:", list(detail.get("comparisons", {})))
                    print("Full numerical comparisons:", folder / "parity-diagnostic.json")
            except (ValueError, OSError) as error:
                print("Could not read final report:", error, flush=True)
    if fallback_archive:
        raise RuntimeError("Runner archive missing; preserve the fallback ZIP/log before rerunning.")
    if not (folder / "report.json").is_file():
        raise RuntimeError("No runner report was produced; preserve the fallback ZIP/log.")
    report = json.loads((folder / "report.json").read_text(encoding="utf-8"))
    if report.get("status") != "passed":
        raise RuntimeError("RQ2 run did not pass; download the ZIP/log for review.")
    if diagnostic:
        if report.get("diagnostic_only") is not True:
            raise RuntimeError("Expected a diagnostic-only report; preserve the ZIP/log.")
        print("Diagnostic completed. Download ZIP/log for review; this does not establish a passed 30-question pilot.", flush=True)
    elif report.get("diagnostic_only"):
        raise RuntimeError("Diagnostic-only output cannot be counted as the full 30-question pilot.")
    return folder


## Run the selected mode once

For the parity failure, enable only `RUN_PARITY_DIAGNOSTIC` in the first cell. The following cell performs the bounded first-question numerical diagnostic and never starts the 30-question pilot. Preserve its ZIP/log and wait for review before running the full panel.

After that review, full-pilot mode compares Brolio, Cookery school and Give up alongside Dominion, Sky mall and Laughter. New failures elsewhere count alongside rescues; generated-answer scoring remains the primary outcome.


In [ ]:
if RUN_PARITY_DIAGNOSTIC:
    rq2_dir = run_rq2_pilot(diagnostic=True)
elif RUN_RQ2_PILOT:
    rq2_dir = run_rq2_pilot()
else:
    raise RuntimeError("Enable one mode in the first cell before running.")


## Download before stopping Kaggle

Download the **result ZIP and its matching console log**, including after a failure. Setup log/runtime JSON are also linked and copied into normal runner evidence. Send the ZIP/log for review before expanding variants or datasets. Save original archives locally under `research/results/`; they are ignored by Git.

To deliberately repeat in the same live session, call `run_rq2_pilot(diagnostic=True)` for another first-question diagnostic or `run_rq2_pilot()` for the full pilot, only after the first process exits and after reviewing any failure. Each call makes a new directory and reloads the model while reusing setup and checkpoint caches. Separate notebooks do not share their virtualenv/session. No baseline change, new environment or RQ1 rerun is needed for this pilot.


In [ ]:
from IPython.display import FileLink, display
paths = [setup_log, runtime_report]
for folder in rq2_runs:
    paths.extend([folder.with_suffix(".zip"), folder.with_suffix(".log")])
for path in paths:
    if path.is_file():
        print(path)
        display(FileLink(str(path)))
